In [1]:
import re
from typing import List, Dict
from PyPDF2 import PdfReader

In [ ]:
def normalize_nepali_text(text: str) -> str:
    """
    Basic normalization for Nepali text
    """
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)  # remove extra spaces
    text = text.strip()

    # Optional: normalize danda to period for consistency
    text = text.replace("।", ".")

    return text


def split_into_sentences(text: str) -> List[str]:
    """
    Split Nepali text into sentences using danda or period
    """
    sentences = re.split(r"[.]", text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences


def chunk_sentences(
    sentences: List[str], chunk_size: int = 300, overlap: int = 50
) -> List[str]:
    """
    Create overlapping chunks from sentences
    chunk_size and overlap are approximate word counts
    """
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        words = sentence.split()
        word_count = len(words)

        if current_length + word_count <= chunk_size:
            current_chunk.append(sentence)
            current_length += word_count
        else:
            # Save current chunk
            chunks.append(" ".join(current_chunk))

            # Create overlap
            overlap_words = []
            if overlap > 0:
                joined = " ".join(current_chunk).split()
                overlap_words = joined[-overlap:]

            current_chunk = [" ".join(overlap_words), sentence]
            current_length = len(overlap_words) + word_count

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


def chunk_pdf(file_path: str, chunk_size: int = 500, overlap: int = 100) -> List[Dict]:
    """
    Main function: takes PDF path and returns chunks with metadata
    """
    reader = PdfReader(file_path)
    all_chunks = []

    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()

        if not text:
            continue

        # Normalize
        text = normalize_nepali_text(text)

        # Sentence split
        sentences = split_into_sentences(text)

        # Chunk
        chunks = chunk_sentences(sentences, chunk_size, overlap)

        # Store with metadata
        for i, chunk in enumerate(chunks):
            all_chunks.append(
                {
                    "content": chunk,
                    "page": page_num + 1,
                    "chunk_id": f"page_{page_num+1}_chunk_{i}",
                }
            )

    return all_chunks

In [4]:
chunks = chunk_pdf("../gen z report.pdf")

print(chunks[0])

{'content': 'प्राक्कथन नेपालमा जहाननयााँ राणा शासनको अवसानपनि 2007 सालमा प्रजातन्त्रको उदय भयो प्रजातन्त्र स्थापनापनि 2046 सालसम्म 22 वटा सरकार बने 2047 सालमा प्रजातन्त्रको पुनस्थाापना, 2052 सालमा माओवादी सशस्त्र नबद्रोह र 2062/2063 सालमा भएको वृहत जनआन्त्दोलनबाट २३८ वर्ा पुरानो राजवंशको अन्त््य समेत भई देशले संघीय लोकतान्त्न्त्रक गणतन्त्रा्मक शासन व्यवस्था अंनगकार गयो 2046 सालदेन्त्ि हालसम्म आइपुग्दा ३५ बर्ामा 30 वटा सरकार बने यस अवनिमा आनथाक ववकासले केही गनत नलएको देन्त्िन्त्ि तर आशानतत सफलता प्राप्त गना सकेन राज्य सत्तामा बारम्बार एउटै अनुहार रहनु तर ववकासका लानग दीघाकालीन योजना कायाान्त्वयनमा न्त्शनथलता हुनु, भ्रष् टाचारको व्यापकता ददनप्रनतददन झांनगदै जानु, श्रोतको ववतरणमा असमानता कायम हुनु जस्ता वियाकलापले देश नभर रोजगारीका अवसर कम भई पनिल्लो समय युवाहरुको वैदेन्त्शक पलायनमा बढोत्तरी हुन पुगेको अवस्थाका कारण देशमा ववकास र सुशासन भन्त्दा पनन सत्ता संघर्ा र सरकार पररवतानको िेल बढी भएको देन्त्िन्त्ि ननवााचनमा उनै व्यन्त्िहरुको पुनरावृन्त्त्त रहनुले कुशल शासकलाई देशको भववष्यको न्त्चन्